In [1]:
# Installation des bibliothèques nécessaires:
!pip install paho-mqtt numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.2/67.2 kB 1.2 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# ===================================================================
# SCRIPT D'ENTRAÎNEMENT COMPLET POUR LE SCÉNARIO 3
# Version finale pour Google Colab avec persistance sur Google Drive
# ===================================================================

# Importation des bibliothèques
import paho.mqtt.client as mqtt
import time
import random
import csv
import sys
import numpy as np
import os
import pickle

# ===================================================================
# BLOC 1 : CONFIGURATION POUR GOOGLE DRIVE
# ===================================================================
# Ce chemin doit correspondre au dossier que vous avez créé sur votre Drive.
try:
    PROJECT_PATH = '/content/drive/MyDrive/Mon_Projet_RL/'
    if not os.path.exists(PROJECT_PATH):
        print(f"ERREUR : Le dossier de projet '{PROJECT_PATH}' est introuvable.")
        print("Veuillez vérifier que votre Google Drive est monté et que le dossier existe.")
        sys.exit()
except Exception as e:
    print(f"Une erreur est survenue lors de la configuration des chemins : {e}")
    sys.exit()
# ===================================================================

# --- CONFIGURATION MQTT ET SUJETS ---
BROKER_ADDRESS = "broker.hivemq.com"
BROKER_PORT = 1883
TOPIC_ARRIVAL = "cupcarbon/linda/simulator/entry"
TOPIC_EXIT = "cupcarbon/linda/simulator/exit"
TOPIC_ACTION = "cupcarbon/linda/simulator/action"
TOPIC_CAR = "cupcarbon/linda/realparking/car"
TOPIC_LIGHT = "cupcarbon/linda/realparking/light"

# --- VARIABLES D'ENVIRONNEMENT ET PARAMÈTRES ---
TOTAL_SPOTS = 150
SPOTS_PER_FLOOR = 50
OCCUPANCY_THRESHOLD = 25
ENERGY_PER_LED_WH_PER_MIN = 0.1
parking_spots_occupied = {}
all_spots = [f"{p}:{s}" for p in range(1, 4) for s in range(1, 51)]
simulation_data = [] # Uniquement pour le rapport final de la dernière simulation
q_table = {}
actions_map = {0: "assign_floor_1", 1: "assign_floor_2", 2: "assign_floor_3"}
num_actions = len(actions_map)
learning_rate = 0.1
discount_factor = 0.9
epsilon = 0.0
last_event_time = None
total_energy_consumed_wh = 0.0
total_reward_current_simulation = 0.0
failed_assignments_count = 0
occupancy_per_floor = {1: 0, 2: 0, 3: 0}
light_status_per_floor = {1: 0, 2: 0, 3: 0}

# ===================================================================
# BLOC 2 : DÉFINITION DES CHEMINS COMPLETS SUR LE DRIVE
# ===================================================================
Q_TABLE_FILE = os.path.join(PROJECT_PATH, 'q_table_final.pkl')
EPSILON_FILE = os.path.join(PROJECT_PATH, 'epsilon_value.pkl')
#TRAINING_PROGRESS_FILE = os.path.join(PROJECT_PATH, 'training_progress.csv')
FINAL_REPORT_FILE = os.path.join(PROJECT_PATH, 'simulation_report_S3_expert.csv')
# ===================================================================

# Chargement de la Q-table depuis le Drive
try:
    if os.path.exists(Q_TABLE_FILE):
        with open(Q_TABLE_FILE, 'rb') as f: q_table = pickle.load(f)
        print(f"🧠 Q-table chargée depuis le Drive ({len(q_table)} états).")
except Exception as e: print(f"⚠️ Q-table non chargée : {e}.")

# Chargement d'epsilon depuis le Drive
try:
    if os.path.exists(EPSILON_FILE):
        with open(EPSILON_FILE, 'rb') as f: epsilon = pickle.load(f)
        print(f"📉 Epsilon chargé depuis le Drive : {epsilon:.4f}")
    else:
        print(f"🆕 Epsilon par défaut : {epsilon:.4f}")
except Exception as e: print(f"⚠️ Epsilon non chargé : {e}.")

# --- FONCTIONS ---

def on_connect(client, userdata, flags, rc, properties):
    if rc == 0:
        print("✅ Connecté à MQTT avec succès.")
        client.subscribe(TOPIC_ARRIVAL)
        client.subscribe(TOPIC_EXIT)
        client.subscribe(TOPIC_ACTION)
        print(f"📡 Abonné aux sujets.")
    else:
        print(f"❌ Échec de la connexion à MQTT. Code: {rc}")
        sys.exit()

def get_state():
    occ1_discrete = min(9, occupancy_per_floor[1] // (SPOTS_PER_FLOOR // 10))
    occ2_discrete = min(9, occupancy_per_floor[2] // (SPOTS_PER_FLOOR // 10))
    occ3_discrete = min(9, occupancy_per_floor[3] // (SPOTS_PER_FLOOR // 10))
    return (occ1_discrete, occ2_discrete, occ3_discrete)

def choose_action(state):
    if state not in q_table:
        q_table[state] = np.zeros(num_actions)
    available_actions_indices = [i for i in range(num_actions) if occupancy_per_floor[i+1] < SPOTS_PER_FLOOR]
    if not available_actions_indices: return None
    if random.uniform(0, 1) < epsilon:
        return random.choice(available_actions_indices)
    else:
        q_values_for_state = q_table[state]
        masked_q_values = {i: q_values_for_state[i] for i in available_actions_indices}
        return max(masked_q_values, key=masked_q_values.get)

def update_q_table(state, action_index, reward, next_state):
    if state not in q_table: q_table[state] = np.zeros(num_actions)
    if next_state not in q_table: q_table[next_state] = np.zeros(num_actions)
    q_value = q_table[state][action_index]
    max_next_q = np.max(q_table[next_state])
    new_q_value = q_value + learning_rate * (reward + discount_factor * max_next_q - q_value)
    q_table[state][action_index] = new_q_value

def get_reward(is_optimal_choice, is_critical_error):
    if is_critical_error: return -50
    return 1 if is_optimal_choice else -10

def update_energy_consumption():
    global last_event_time, total_energy_consumed_wh
    if last_event_time is None:
        last_event_time = time.time()
        return
    time_elapsed_minutes = (time.time() - last_event_time) / 60
    leds_on = sum((SPOTS_PER_FLOOR - occupancy_per_floor[floor]) for floor in range(1, 4) if light_status_per_floor[floor] == 1)
    energy_consumed_interval = leds_on * ENERGY_PER_LED_WH_PER_MIN * time_elapsed_minutes
    total_energy_consumed_wh += energy_consumed_interval
    last_event_time = time.time()

def update_training_metrics():
    csv_filename = TRAINING_PROGRESS_FILE
    file_exists = os.path.isfile(csv_filename) and os.path.getsize(csv_filename) > 0
    with open(csv_filename, mode='a', newline='') as file:
        writer = csv.writer(file)
        if not file_exists:
            writer.writerow(['simulation_id', 'total_reward', 'total_energy_wh', 'failed_assignments', 'occupancy_variance', 'epsilon'])

        last_id = 0
        if file_exists:
            with open(csv_filename, 'r') as f_read:
                lines = f_read.readlines()
                if len(lines) > 1: last_id = int(lines[-1].split(',')[0])

        occupancy_variance = np.var(list(occupancy_per_floor.values()))
        writer.writerow([last_id + 1, f'{total_reward_current_simulation:.2f}', f'{total_energy_consumed_wh:.2f}', failed_assignments_count, f'{occupancy_variance:.2f}', f'{epsilon:.4f}'])
    print(f"📊 Métriques de l'épisode #{last_id + 1} sauvegardées sur Google Drive.")

def toggle_floor_lights(client, floor, turn_on):
    action_str = "Allumage" if turn_on else "Extinction"
    print(f"  💡 {action_str} des LED pour l'étage {floor}.")
    for s in range(1, SPOTS_PER_FLOOR + 1):
        spot_id = f"{floor}:{s}"
        if spot_id not in parking_spots_occupied:
            status = 1 if turn_on else 0
            client.publish(TOPIC_LIGHT, f"{floor}:{s}:{status}")

def on_message(client, userdata, msg):
    global parking_spots_occupied, total_reward_current_simulation, failed_assignments_count, epsilon, total_energy_consumed_wh, simulation_data

    update_energy_consumption()
    payload = msg.payload.decode()

    if msg.topic == TOPIC_ARRIVAL:
        current_state = get_state()
        action_index = choose_action(current_state)
        if action_index is None:
            failed_assignments_count += 1
            return

        assigned_floor = action_index + 1
        valid_floors = {k:v for k,v in occupancy_per_floor.items() if v < SPOTS_PER_FLOOR}
        optimal_floor = min(valid_floors, key=valid_floors.get) if valid_floors else assigned_floor
        is_optimal_choice = (assigned_floor == optimal_floor)

        available_spots_on_floor = [s for s in all_spots if s.startswith(f"{assigned_floor}:") and s not in parking_spots_occupied]
        spot_id = random.choice(available_spots_on_floor)
        parking_spots_occupied[spot_id] = payload
        occupancy_per_floor[assigned_floor] += 1
        platform, place = spot_id.split(':')
        client.publish(TOPIC_CAR, f"{platform}:{place}:1")
        client.publish(TOPIC_LIGHT, f"{platform}:{place}:0")

        reward = get_reward(is_optimal_choice, is_critical_error=False)
        total_reward_current_simulation += reward
        next_state = get_state()
        #update_q_table(current_state, action_index, reward, next_state)

        if occupancy_per_floor[assigned_floor] >= OCCUPANCY_THRESHOLD and light_status_per_floor[assigned_floor] == 0:
            toggle_floor_lights(client, assigned_floor, turn_on=True)
            light_status_per_floor[assigned_floor] = 1

        simulation_data.append({'Timestamp': time.strftime('%Y-%m-%d %H:%M:%S'), 'Event Type': 'ARRIVAL', 'Assigned Spot': spot_id})

    elif msg.topic == TOPIC_EXIT:
        spot_id = next((spot for spot, car in parking_spots_occupied.items() if car == payload), None)
        if not spot_id: return
        platform, place = map(int, spot_id.split(':'))
        del parking_spots_occupied[spot_id]
        occupancy_per_floor[platform] -= 1
        client.publish(TOPIC_CAR, f"{platform}:{place}:0")
        light_status = 1 if light_status_per_floor[platform] == 1 else 0
        client.publish(TOPIC_LIGHT, f"{platform}:{place}:{light_status}")
        if occupancy_per_floor[platform] < OCCUPANCY_THRESHOLD and light_status_per_floor[platform] == 1:
            toggle_floor_lights(client, platform, turn_on=False)
            light_status_per_floor[platform] = 0

        simulation_data.append({'Timestamp': time.strftime('%Y-%m-%d %H:%M:%S'), 'Event Type': 'EXIT', 'Assigned Spot': spot_id})

    elif msg.topic == TOPIC_ACTION and payload == "STOP_SIMULATION":
        print("\n\n🏁 Message de fin de simulation reçu.")
        update_energy_consumption()
        simulation_duration_minutes = 10
        #update_training_metrics()

        #epsilon = max(0.1, epsilon * 0.99)
        #print(f"📉 Epsilon mis à jour : {epsilon:.4f}")

        # Sauvegarde de la mémoire sur le Drive
        try:
            with open(Q_TABLE_FILE, 'wb') as f: pickle.dump(q_table, f)
            print(f"💾 Q-table sauvegardée sur Google Drive.")
        except Exception as e: print(f"❌ Erreur sauvegarde Q-table : {e}")
        try:
            with open(EPSILON_FILE, 'wb') as f: pickle.dump(epsilon, f)
            print(f"💾 Epsilon sauvegardé sur Google Drive.")
        except Exception as e: print(f"❌ Erreur sauvegarde epsilon : {e}")

        # Écriture d'un rapport final pour cet épisode (qui sera écrasé au prochain)
        try:
            with open(FINAL_REPORT_FILE, mode='w', newline='') as file:
                writer = csv.writer(file)
                writer.writerow(['Timestamp', 'Event Type', 'Assigned Spot'])
                for row in simulation_data:
                    writer.writerow([row['Timestamp'], row['Event Type'], row['Assigned Spot']])
                # Écrire le bloc de résumé, au même format que S1 et S2
                writer.writerow([]) # Ligne vide pour la séparation
                writer.writerow(['--- Rapport Final ---'])
                writer.writerow(['Durée de la simulation', f'{simulation_duration_minutes} minutes'])
                writer.writerow(['Consommation d\'énergie totale des LED', f'{total_energy_consumed_wh:.2f} Wh'])
                writer.writerow(['Nombre d\'attributions échouées', failed_assignments_count])

            print(f"📊 Rapport de simulation (avec résumé) sauvegardé dans '{FINAL_REPORT_FILE}'.")

        except Exception as e:
            print(f"❌ Erreur lors de l'écriture du fichier CSV : {e}")
        # ===================================================================

        client.loop_stop()
        client.disconnect()
        print("Déconnecté. Fin de l'évaluation.")

# --- PROGRAMME PRINCIPAL ---
if __name__ == "__main__":
    client = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2)
    client.on_connect = on_connect
    client.on_message = on_message
    try:
        client.connect(BROKER_ADDRESS, BROKER_PORT, 60)
        client.loop_forever()
    except KeyboardInterrupt:
        print("\nℹ️ Arrêt du script.")
    finally:
        if client.is_connected(): client.disconnect()
        print("Déconnecté.")

🧠 Q-table chargée depuis le Drive (254 états).
📉 Epsilon chargé depuis le Drive : 0.2675
✅ Connecté à MQTT avec succès.
📡 Abonné aux sujets.
  💡 Allumage des LED pour l'étage 2.
  💡 Allumage des LED pour l'étage 3.
  💡 Allumage des LED pour l'étage 1.
  💡 Extinction des LED pour l'étage 1.
  💡 Allumage des LED pour l'étage 1.
  💡 Extinction des LED pour l'étage 3.
  💡 Allumage des LED pour l'étage 3.
  💡 Extinction des LED pour l'étage 1.
  💡 Extinction des LED pour l'étage 3.
  💡 Extinction des LED pour l'étage 2.


🏁 Message de fin de simulation reçu.
💾 Q-table sauvegardée sur Google Drive.
💾 Epsilon sauvegardé sur Google Drive.
📊 Rapport de simulation (avec résumé) sauvegardé dans '/content/drive/MyDrive/Mon_Projet_RL/simulation_report_S3_expert.csv'.
Déconnecté. Fin de l'évaluation.
Déconnecté.
